In [3]:

import pandas as pd
import numpy as np
from pathlib import Path


ORS_XLSX_PATH = Path("ORS (2015_2017_2019) copied 2025-7-5.xlsx")  
SHEET_NAME = "HIV2015"
OUTPUT_CSV = Path("impact_score.csv")


DRUG_COLS = ['3TC','ABC','AZT','ddl','d4T','EFV','FTC','LPV/r','NVP','TDF','ATV/r']

print("ORS file exists:", ORS_XLSX_PATH.exists())


ORS file exists: True


In [4]:

df_raw = pd.read_excel(ORS_XLSX_PATH, sheet_name=SHEET_NAME, header=None, engine="openpyxl")
df_raw.shape, df_raw.head()


((238, 84),
             0           1           2                                3   \
 0          NaN         NaN         NaN                              NaN   
 1          NaN         NaN         NaN                              NaN   
 2      Country  WHO Region  Population              Geographical Region   
 3          NaN         NaN         NaN                              NaN   
 4  Afghanistan         EMR    33736494  East, South and South-East Asia   
 
           4          5            6               7   \
 0        NaN          1          NaN             NaN   
 1        NaN        NaN          NaN             NaN   
 2  WHO Group       DALY  Adult DALYs  Children DALYs   
 3        NaN        NaN          NaN             NaN   
 4          A  10752.548     9224.366        1528.182   
 
                            8                           9   ...        74  \
 0                         NaN                         NaN  ...       NaN   
 1                         NaN  

In [5]:

# Column indices in the HIV2015 sheet (0-based)
COL_COUNTRY = 0
COL_ADULT_DALYS = 6
COL_CHILD_DALYS = 7
COL_RR_OVERALL = 8           # overall retention rate (%)

COL_ADULT_COV = 16           # adult coverage (proportion)
COL_CHILD_COV = 19           # child coverage (proportion)

# Regimen table columns (0-based) – see HIV2015 sheet
COL_REGIMEN = 44
COL_ADULT_PROP = 45
COL_ADULT_EFF = 46
COL_CHILD_PROP = 47
COL_CHILD_EFF = 48
COL_N_DRUGS = 49

# Cells holding the overall first/second line shares (row, col) in 0-based indexing
CELL_ADULT_FIRST_SHARE  = (4, 41)
CELL_ADULT_SECOND_SHARE = (5, 41)
CELL_CHILD_FIRST_SHARE  = (9, 41)
CELL_CHILD_SECOND_SHARE = (10, 41)

# Pull those line shares from the sheet
adult_first_share  = df_raw.iat[CELL_ADULT_FIRST_SHARE[0],  CELL_ADULT_FIRST_SHARE[1]]
adult_second_share = df_raw.iat[CELL_ADULT_SECOND_SHARE[0], CELL_ADULT_SECOND_SHARE[1]]
child_first_share  = df_raw.iat[CELL_CHILD_FIRST_SHARE[0],  CELL_CHILD_FIRST_SHARE[1]]
child_second_share = df_raw.iat[CELL_CHILD_SECOND_SHARE[0], CELL_CHILD_SECOND_SHARE[1]]

adult_first_share, adult_second_share, child_first_share, child_second_share


(0.90835, 0.09165, 0.89325, 0.10675)

## 3) Helper functions

These helpers implement the model rules used in the provided materials:

- **`safe_float`**: spreadsheet cells can be blank or strings; convert safely.
- **`treatment_years`**: average treatment period  
  \( Years = \frac{100}{100 - RR} \) and RR is **capped** (97.14) per the model note.
- **`parse_regimen_drugs`**: convert a regimen label into a list of drug codes.


In [6]:

def safe_float(x, default=0.0) -> float:
    """Convert a spreadsheet cell value to float; blanks/NaN become `default`."""
    if x is None:
        return default
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return default
        try:
            return float(x)
        except Exception:
            return default
    if pd.isna(x):
        return default
    try:
        return float(x)
    except Exception:
        return default


def treatment_years(rr_pct: float) -> float:
    """Average treatment period (years) = 100/(100 - RR).

    The retention rate is capped at 97.14 per model notes.
    """
    rr_pct = min(float(rr_pct), 97.14)
    return 100.0 / (100.0 - rr_pct) if rr_pct < 100 else float("inf")


def parse_regimen_drugs(regimen: str):
    """Split regimen string into drug codes.

    Special handling matches the ORS sheet aggregation behavior:
    - 'Others TDF...' is treated as only 'TDF' (single-drug attribution bucket)
    - 'Others' alone contributes no specific drug
    """
    regimen = str(regimen).strip()

    if regimen.lower().startswith("others tdf"):
        return ["TDF"]
    if regimen.lower() == "others":
        return []

    return [p.strip() for p in regimen.split("+") if p.strip()]


##  Build the regimen table (first-line + second-line)

The HIV2015 sheet contains a regimen block with:
- regimen name (`TDF+3TC+EFV`, etc.)
- adult/child proportion in that regimen
- adult/child effectiveness for that regimen
- optional `n_drugs` count

We build a tidy regimen table so we can iterate cleanly in Python.


In [7]:

def regimen_n_drugs(reg_row: pd.Series) -> int:
    """Determine the number of drugs to split regimen attribution across.

    If the sheet provides `n_drugs`, we use it; otherwise we infer from regimen text.
    """
    v = reg_row.get("n_drugs", np.nan)
    if pd.notna(v):
        return int(v)

    # A conservative fallback: treat 'Others' buckets as single-drug attribution
    if isinstance(reg_row.get("regimen", ""), str) and "Others" in reg_row["regimen"]:
        return 1

    drugs = reg_row.get("drugs2", [])
    return max(1, len(drugs))


def build_regimen_table(df_raw: pd.DataFrame,
                        adult_first_share: float, adult_second_share: float,
                        child_first_share: float, child_second_share: float) -> pd.DataFrame:
    """Extract regimen rows from the HIV2015 sheet and compute X terms.

    X terms (per regimen) are:
      x_adult = line_share * regimen_prop_adult * regimen_eff_adult
      x_child = line_share * regimen_prop_child * regimen_eff_child
    """
    # Identify regimen rows (strings containing '+' or 'Others')
    reg_rows = []
    for r in range(df_raw.shape[0]):
        v = df_raw.iat[r, COL_REGIMEN]
        if isinstance(v, str) and (("+" in v) or ("Others" in v)) and ("Regimens" not in v):
            reg_rows.append(r)

    regimens = []
    for r in reg_rows:
        # The HIV2015 sheet places first-line regimens earlier, second-line later.
        # The cutoff below matches the reference workflow for this ORS workbook.
        line = "first" if r < 18 else "second"

        regimens.append({
            "line": line,
            "regimen": df_raw.iat[r, COL_REGIMEN],
            "adult_prop": safe_float(df_raw.iat[r, COL_ADULT_PROP]),
            "adult_eff":  safe_float(df_raw.iat[r, COL_ADULT_EFF]),
            "child_prop": safe_float(df_raw.iat[r, COL_CHILD_PROP]),
            "child_eff":  safe_float(df_raw.iat[r, COL_CHILD_EFF]),
            "n_drugs":    safe_float(df_raw.iat[r, COL_N_DRUGS], default=np.nan),
        })

    reg_df = pd.DataFrame(regimens)
    reg_df["drugs2"] = reg_df["regimen"].apply(parse_regimen_drugs)

    adult_share = {"first": safe_float(adult_first_share), "second": safe_float(adult_second_share)}
    child_share = {"first": safe_float(child_first_share), "second": safe_float(child_second_share)}

    reg_df["x_adult"] = reg_df.apply(lambda r: adult_share[r["line"]] * r["adult_prop"] * r["adult_eff"], axis=1)
    reg_df["x_child"] = reg_df.apply(lambda r: child_share[r["line"]] * r["child_prop"] * r["child_eff"], axis=1)

    return reg_df


reg_df = build_regimen_table(df_raw, adult_first_share, adult_second_share, child_first_share, child_second_share)
reg_df.head(10)


,line,regimen,adult_prop,adult_eff,child_prop,child_eff,n_drugs,drugs2,x_adult,x_child
0,first,TDF + 3TC + EFV,0.324,0.780000,0.00,0.788333,3.0,"[TDF, 3TC, EFV]",0.229558,0.000000
1,first,TDF + FTC + EFV,0.224,0.771429,0.00,0.798947,3.0,"[TDF, FTC, EFV]",0.156963,0.000000
2,first,AZT + 3TC + NVP,0.201,0.819333,0.43,0.819333,3.0,"[AZT, 3TC, NVP]",0.149593,0.314704
3,first,TDF + 3TC + NVP,0.131,0.750000,0.00,0.655000,3.0,"[TDF, 3TC, NVP]",0.089245,0.000000
4,first,AZT + 3TC + EFV,0.085,0.730000,0.08,0.706667,3.0,"[AZT, 3TC, EFV]",0.056363,0.050498
5,first,Others TDF based,0.008,0.751247,0.00,0.500000,1.0,[TDF],0.005459,0.000000
6,first,Others,0.027,0.751247,0.02,0.500000,NaN,[],0.018425,0.008932
7,first,ABC + 3TC + LPV/r,0.000,0.630000,0.14,0.630000,3.0,"[ABC, 3TC, LPV/r]",0.000000,0.078785
8,first,ABC + 3TC + EFV,0.000,0.694750,0.13,0.729800,3.0,"[ABC, 3TC, EFV]",0.000000,0.084746
9,first,ABC + 3TC + NVP,0.000,0.751247,0.07,0.500000,3.0,"[ABC, 3TC, NVP]",0.000000,0.031264


In [8]:

def compute_country_impacts(df_raw: pd.DataFrame, reg_df: pd.DataFrame, row_idx: int) -> dict:
    """Compute per-drug impacts for one country row."""
    adult_dalys = safe_float(df_raw.iat[row_idx, COL_ADULT_DALYS])
    child_dalys = safe_float(df_raw.iat[row_idx, COL_CHILD_DALYS])
    adult_cov   = safe_float(df_raw.iat[row_idx, COL_ADULT_COV])
    child_cov   = safe_float(df_raw.iat[row_idx, COL_CHILD_COV])

    years = treatment_years(safe_float(df_raw.iat[row_idx, COL_RR_OVERALL], default=97.14))

    impacts = {d: 0.0 for d in DRUG_COLS}

    for _, reg in reg_df.iterrows():
        nd = regimen_n_drugs(reg)
        xa = safe_float(reg["x_adult"])
        xc = safe_float(reg["x_child"])

        # Avoid division by zero / invalid denominator
        imp_ad = adult_dalys * adult_cov * xa / (1 - adult_cov * xa) / nd / years if (xa and adult_cov) else 0.0
        imp_ch = child_dalys * child_cov * xc / (1 - child_cov * xc) / nd / years if (xc and child_cov) else 0.0

        contrib = imp_ad + imp_ch

        for drug in reg["drugs2"]:
            if drug in impacts:
                impacts[drug] += contrib

    return impacts


In [9]:

# Detect country rows
country_rows = []
for r in range(df_raw.shape[0]):
    c = df_raw.iat[r, COL_COUNTRY]
    if isinstance(c, str) and c.strip() and c.strip().lower() != "country":
        # column 5 is used as a numeric sentinel in this workbook layout
        if pd.notna(df_raw.iat[r, 5]) and isinstance(df_raw.iat[r, 5], (int, float, np.number)):
            country_rows.append(r)

len(country_rows), country_rows[:5]


(217, [4, 5, 6, 7, 8])

In [10]:

# Compute per-country impacts
rows_out = []
for r in country_rows:
    country = str(df_raw.iat[r, COL_COUNTRY]).strip()
    impacts = compute_country_impacts(df_raw, reg_df, r)
    rows_out.append({"Country": country, **impacts, "Overall Treatment Impact": sum(impacts.values())})

out_df = pd.DataFrame(rows_out)
out_df.head()


,Country,3TC,ABC,AZT,ddl,d4T,EFV,FTC,LPV/r,NVP,TDF,ATV/r,Overall Treatment Impact
0,Afghanistan,35.667802,3.483114,15.831563,0.102642,0.523836,21.363380,7.057698,4.612259,16.425901,23.749159,0.412628,129.229982
1,Albania,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,Algeria,186.166832,3.714239,70.801802,0.088267,0.455925,140.014900,49.132222,15.754865,77.092066,164.992819,2.513475,710.727413
3,American Samoa,0.054960,0.000219,0.020441,0.000000,0.000000,0.042521,0.015478,0.004611,0.022444,0.051345,0.000863,0.212881
4,Andorra,0.176927,0.000706,0.065802,0.000000,0.000000,0.136883,0.049826,0.014843,0.072249,0.165287,0.002777,0.685301


In [11]:

expected_cols = ["Country"] + DRUG_COLS + ["Overall Treatment Impact"]
missing = [c for c in expected_cols if c not in out_df.columns]
print("Missing columns:", missing)

print("Countries:", out_df["Country"].nunique())
print("Any negative values?", (out_df[DRUG_COLS + ["Overall Treatment Impact"]] < 0).any().any())

out_df.describe(include="all").transpose().head(15)


Missing columns: []
Countries: 217
Any negative values? False


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Country,217,217,Afghanistan,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3TC,217.0,NaN,NaN,NaN,3348.1734,15238.76382,0.0,0.0,13.460479,426.333301,174711.665995
ABC,217.0,NaN,NaN,NaN,289.420684,1760.407016,0.0,0.0,0.066535,10.392638,21562.330876
AZT,217.0,NaN,NaN,NaN,1485.010576,7115.133013,0.0,0.0,5.006209,171.854463,82863.264823
ddl,217.0,NaN,NaN,NaN,8.224933,50.697647,0.0,0.0,0.0,0.197616,623.416758
d4T,217.0,NaN,NaN,NaN,42.280334,260.584322,0.0,0.0,0.0,1.0105,3202.534516
EFV,217.0,NaN,NaN,NaN,2050.287228,8579.227499,0.0,0.0,10.048168,323.429793,94559.133013
FTC,217.0,NaN,NaN,NaN,676.311139,2722.848249,0.0,0.0,3.492806,112.827032,29259.120978
LPV/r,217.0,NaN,NaN,NaN,395.837145,1983.203839,0.0,0.0,1.129264,41.397143,23538.031717
NVP,217.0,NaN,NaN,NaN,1548.671403,7316.128302,0.0,0.0,5.496692,180.47193,84823.092003


In [12]:

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {OUTPUT_CSV.resolve()} with {len(out_df)} countries.")


Wrote /Users/tarashbudhrani/Global Impact Health /impact_score.csv with 217 countries.


In [15]:
final_df = results.copy()

final_df.to_csv("impact_score.csv", index=False)

print("Total rows:", len(final_df))
print("Unique countries:", final_df['Country'].nunique())
print("Unique drugs:", final_df['Drug'].nunique())
print(final_df.head())

NameError: name 'results' is not defined